In [ ]:
api_key = "####YOUR_API_KEY####"

In [26]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import re
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from sklearn.manifold import TSNE
from tqdm import tqdm
import plotly.express as px
import requests
import time
import json

In [ ]:
newsdata = pd.read_csv('path/to/real-world-nonparallel.csv')

In [7]:
newsdata.head(2)

,title,link,creator,description,pubDate,pubDateTZ,source_id,source_name,source_url,source_icon,language,country,category,x,y,translated_title,translated_description,translated_title_embeddings
0,CMHO कार्यालय का अतिरिक्त कार्यभार बना चर्चा म...,https://www.bhaskar.com/local/rajasthan/sawai-...,NaN,अपनों पर सितम गैरों पर रहम की कहानी सवाई माधोप...,2025-02-13 09:21:00,UTC,bhaskar_hindi,Bhaskar,https://www.bhaskar.com,https://i.bytvi.com/domain_icons/bhaskar_hindi...,hindi,['india'],"['domestic', 'other']",-45.92981,3.840987,Additional workload of the CMHO office in disc...,"On the sidelines, the story of Rahm appears to...","[-0.0024131080135703087, -0.012002086266875267..."
1,Crude Oil Prices Decline as US-Russia Talks Ra...,https://in.investing.com/analysis/crude-oil-pr...,['ING Economic and Financial Analysis'],NaN,2025-02-13 09:21:00,UTC,investing_in,Investing India,https://in.investing.com,NaN,english,['india'],['business'],-19.17778,5.973638,Crude Oil Prices Decline as US-Russia Talks Ra...,NaN,"[-0.014294152148067951, 0.026426952332258224, ..."


In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [17]:
tqdm.pandas()

# language codes map : https://cloud.google.com/translate/docs/languages
lang_map = {
    "meitei": "mni-Mtei", "oriya": "or", "hindi": "hi", "tamil": "ta", "bengali": "bn", "urdu": "ur",
    "assamese": "as", "marathi": "mr", "telugu": "te", "gujarati": "gu", "punjabi": "pa",
    "kannada": "kn", "malayalam": "ml"
}

# Translation using Google Cloud Translation API
def translate_to_english(text, source_lang_code, api_key):
    try:
        if not isinstance(text, str) or text.strip() == "" or source_lang_code == "en":
            return text

        url = "https://translation.googleapis.com/language/translate/v2"
        params = {
            "q": text,
            "source": source_lang_code,
            "target": "en",
            "key": api_key,
            "format": "text"
        }

        response = requests.post(url, data=params)
        if response.status_code == 200:
            return response.json()["data"]["translations"][0]["translatedText"]
        else:
            print(f"Error: {response.status_code} - {response.text}")
            return text
    except Exception as e:
        print(f"Translation error from {source_lang_code}: {text[:50]}..., Error: {e}")
        return text

In [18]:
def translate(row, api_key):
    lang = row['language']
    text = row['title']
    if lang != 'english' and lang in lang_map:
        return translate_to_english(text, lang_map[lang], api_key)
    return text


In [ ]:
newsdata_translated = newsdata.copy()
newsdata_translated['translated_title'] = newsdata.progress_apply(lambda row: translate(row, api_key), axis=1)

100%|██████████| 13257/13257 [14:49<00:00, 14.91it/s]


`newsdata_translated.csv` is saved in the repository and can help to avoid redoing the translations.

In [23]:
newsdata_translated.head()

,title,link,creator,description,pubDate,pubDateTZ,source_id,source_name,source_url,source_icon,language,country,category,translated_title
0,CMHO कार्यालय का अतिरिक्त कार्यभार बना चर्चा म...,https://www.bhaskar.com/local/rajasthan/sawai-...,NaN,अपनों पर सितम गैरों पर रहम की कहानी सवाई माधोप...,2025-02-13 09:21:00,UTC,bhaskar_hindi,Bhaskar,https://www.bhaskar.com,https://i.bytvi.com/domain_icons/bhaskar_hindi...,hindi,['india'],"['domestic', 'other']",Additional charge of CMHO office became a topi...
1,Crude Oil Prices Decline as US-Russia Talks Ra...,https://in.investing.com/analysis/crude-oil-pr...,['ING Economic and Financial Analysis'],NaN,2025-02-13 09:21:00,UTC,investing_in,Investing India,https://in.investing.com,NaN,english,['india'],['business'],Crude Oil Prices Decline as US-Russia Talks Ra...
2,বিয়া চলি থকাৰ সময়তেই উপস্থিত আৰক্ষী; বিবাহথল...,https://www.assamtv9.com/assam/groom-and-bride...,['Raj Saikia'],আলহী-অতিথিৰ সৈতে ৰজনজনাই থকা বিয়াঘৰত হঠাৎ উপস...,2025-02-13 09:21:00,UTC,assamtv9,Assamese News Today,https://www.assamtv9.com,https://i.bytvi.com/domain_icons/assamtv9.png,assamese,['india'],['top'],Police arrive at the wedding; The bride and gr...
3,Aero India 2025: ಪ್ರಮುಖ ನೌಕಾ ವಾಯುಯಾನ ತಂತ್ರಜ್ಞಾ...,https://www.kannadaprabha.com/karnataka/2025/F...,['Sumana Upadhyaya'],NaN,2025-02-13 09:20:54,UTC,kannadaprabha,Kannada Prabha Online,https://www.kannadaprabha.com,NaN,kannada,['india'],['top'],Aero India 2025: Navy opens up opportunities f...
4,Health experts and enthusiasts react to OTT's ...,https://timesofindia.indiatimes.com/life-style...,NaN,"The Netflix show Apple Cider Vinegar, starring...",2025-02-13 09:20:54,UTC,toi,The Times Of India,https://timesofindia.indiatimes.com,https://i.bytvi.com/domain_icons/toi.png,english,['india'],['top'],Health experts and enthusiasts react to OTT's ...


# Generate Embeddings using MPNet

In [ ]:
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
model = model.to(device)

In [25]:
# Generate embeddings
translated_title_embeddings = []
with tqdm(total=len(newsdata_translated), desc="Generating Title Embeddings", dynamic_ncols=True, leave=True) as pbar:
    for title in newsdata_translated["translated_title"].astype(str):
        embedding = model.encode(title, convert_to_tensor=True, device=device).cpu().numpy()  # Move to CPU
        translated_title_embeddings.append(embedding.tolist())  # Convert NumPy array to list for JSON storage
        pbar.update(1)

# Add embeddings to DataFrame
newsdata_translated["translated_title_embeddings"] = translated_title_embeddings

Generating Title Embeddings: 100%|██████████| 13257/13257 [03:21<00:00, 65.86it/s]


# do dimensionality reduction using t-SNE

In [29]:
embeddings = np.vstack(newsdata_translated["translated_title_embeddings"].values)

# Run t-SNE
tsne = TSNE(
    n_components=2,
    perplexity=50,
    max_iter=2000,
    learning_rate=200,
    metric="cosine",
    random_state=42
)

tsne_results = tsne.fit_transform(embeddings)

## Semantic Representation Map Coloured by Language

In [31]:
newsdata_translated["x"] = tsne_results[:, 0]
newsdata_translated["y"] = tsne_results[:, 1]

# Plot
fig = px.scatter(
    newsdata_translated, x="x", y="y", color="language",
    hover_data=["title", "translated_title", "source_name", "language"],
    title="t-SNE Clustering of News Articles"
)
fig.show()